# Exercise 1.3: Classification and Clustering on a Small Spatial Dataset

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/01_machine_learning/notebooks/exercise_1_3_classification_clustering.ipynb)

This notebook follows the second machine learning lecture. We use one small synthetic geospatial dataset and compare several methods from the slides:

- supervised classification: logistic regression, SVM, decision tree, random forest, and gradient boosting
- unsupervised clustering: K-Means, hierarchical clustering, Gaussian Mixture Models, and DBSCAN
- model evaluation with accuracy, macro F1, confusion matrices, silhouette score, and adjusted Rand index
- side-by-side visual comparison of clustering results on the same data

The dataset is synthetic, but it is designed like a small urban analysis task: each point is a sampled location with local coordinates and simple area attributes. The class label is only used for supervised learning and for checking clustering afterwards.


## 0. Setup

Run this first in Colab. The notebook generates the data locally, so no external dataset download is needed.


In [ ]:
%pip -q install scikit-learn pandas matplotlib seaborn

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    f1_score,
    silhouette_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## 1. Create a small spatial dataset

The points represent locations in a small city area. The coordinate system is local and measured in kilometers from a fictional city center.

The four classes are deliberately not perfectly separated. This makes the comparison between linear and non-linear classifiers more useful.


In [ ]:
def make_urban_sample(random_state=42):
    rng = np.random.default_rng(random_state)
    rows = []

    specs = [
        {
            "zone_type": "Residential",
            "n": 42,
            "mean": (-2.2, -1.1),
            "cov": [[0.35, 0.10], [0.10, 0.22]],
            "road": 3.8,
            "green": 0.32,
            "transit": 0.42,
        },
        {
            "zone_type": "Business center",
            "n": 42,
            "mean": (0.0, 0.0),
            "cov": [[0.28, -0.04], [-0.04, 0.24]],
            "road": 7.4,
            "green": 0.10,
            "transit": 0.86,
        },
        {
            "zone_type": "Green / recreation",
            "n": 38,
            "mean": (1.9, 1.55),
            "cov": [[0.42, 0.06], [0.06, 0.28]],
            "road": 1.9,
            "green": 0.78,
            "transit": 0.28,
        },
    ]

    for spec in specs:
        points = rng.multivariate_normal(spec["mean"], spec["cov"], size=spec["n"])
        for x, y in points:
            distance = np.sqrt(x**2 + y**2)
            rows.append(
                {
                    "x_km": x,
                    "y_km": y,
                    "distance_to_center_km": distance,
                    "road_density": np.clip(rng.normal(spec["road"], 0.45), 0, None),
                    "green_share": np.clip(rng.normal(spec["green"], 0.07), 0, 1),
                    "transit_access": np.clip(rng.normal(spec["transit"], 0.08), 0, 1),
                    "zone_type": spec["zone_type"],
                }
            )

    # A corridor-like class creates a less spherical shape. This is useful for testing SVMs and DBSCAN.
    t = rng.uniform(-2.7, 2.6, size=42)
    x = t + rng.normal(0, 0.24, size=t.size)
    y = 0.52 * t + 0.55 + rng.normal(0, 0.22, size=t.size)
    for xi, yi in zip(x, y):
        distance = np.sqrt(xi**2 + yi**2)
        rows.append(
            {
                "x_km": xi,
                "y_km": yi,
                "distance_to_center_km": distance,
                "road_density": np.clip(rng.normal(6.0, 0.55), 0, None),
                "green_share": np.clip(rng.normal(0.18, 0.06), 0, 1),
                "transit_access": np.clip(rng.normal(0.92, 0.05), 0, 1),
                "zone_type": "Transit corridor",
            }
        )

    return pd.DataFrame(rows).sample(frac=1, random_state=random_state).reset_index(drop=True)

urban_df = make_urban_sample(RANDOM_STATE)
display(urban_df.head())
print(urban_df["zone_type"].value_counts())


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=urban_df,
    x="x_km",
    y="y_km",
    hue="zone_type",
    palette="Set2",
    s=55,
    edgecolor="white",
    linewidth=0.5,
    ax=ax,
)
ax.set_title("Small synthetic urban dataset")
ax.set_xlabel("Local x coordinate (km)")
ax.set_ylabel("Local y coordinate (km)")
ax.legend(title="Known class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


### Student task

Look at the scatter plot and answer briefly:

- Which classes look easy to separate by location only?
- Which classes overlap?
- Which class shape is least compatible with a simple centroid-based method?


## 2. Train/test split for classification

We start with the two spatial coordinates only. This keeps the decision boundaries easy to visualize.

Later you can add the attribute columns and check whether the metrics improve.


In [ ]:
spatial_features = ["x_km", "y_km"]
attribute_features = ["distance_to_center_km", "road_density", "green_share", "transit_access"]

X = urban_df[spatial_features]
y = urban_df["zone_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.28,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")


## 3. Compare several classification methods

The selected methods correspond to the lecture topics:

- logistic regression as a simple linear baseline
- SVM with an RBF kernel for non-linear boundaries
- decision tree as an interpretable rule-based model
- random forest as a bagging ensemble of decision trees
- gradient boosting as a sequential tree ensemble


In [ ]:
classifiers = {
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    ),
    "SVM (RBF kernel)": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=3.0, gamma="scale", random_state=RANDOM_STATE),
    ),
    "Decision tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=6,
        random_state=RANDOM_STATE,
    ),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

classification_results = []
trained_classifiers = {}

for name, model in classifiers.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    trained_classifiers[name] = model
    classification_results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "macro_f1": f1_score(y_test, y_pred, average="macro"),
        }
    )

results_df = pd.DataFrame(classification_results).sort_values("macro_f1", ascending=False)
display(results_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results_long = results_df.melt(id_vars="model", value_vars=["accuracy", "macro_f1"])
sns.barplot(data=results_long, x="value", y="model", hue="variable", ax=ax)
ax.set_xlim(0, 1)
ax.set_xlabel("Score")
ax.set_ylabel("")
ax.set_title("Classification performance on the test set")
plt.show()


## 4. Visualize classification boundaries

A high score is useful, but the decision surface explains what the classifier learned. Compare especially the linear baseline, SVM, tree, and ensemble methods.


In [ ]:
def plot_decision_boundaries(models, X_train, y_train, X_test, y_test):
    class_names = sorted(y_train.unique())
    class_to_id = {label: i for i, label in enumerate(class_names)}

    x_min, x_max = urban_df["x_km"].min() - 0.5, urban_df["x_km"].max() + 0.5
    y_min, y_max = urban_df["y_km"].min() - 0.5, urban_df["y_km"].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 280), np.linspace(y_min, y_max, 240))
    grid = pd.DataFrame({"x_km": xx.ravel(), "y_km": yy.ravel()})

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
    axes = axes.ravel()

    for ax, (name, model) in zip(axes, models.items()):
        pred = model.predict(grid)
        zz = np.array([class_to_id[p] for p in pred]).reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=np.arange(len(class_names) + 1) - 0.5, cmap="Set2", alpha=0.32)
        sns.scatterplot(
            x=X_train["x_km"],
            y=X_train["y_km"],
            hue=y_train,
            hue_order=class_names,
            palette="Set2",
            s=28,
            linewidth=0,
            alpha=0.75,
            legend=False,
            ax=ax,
        )
        sns.scatterplot(
            x=X_test["x_km"],
            y=X_test["y_km"],
            hue=y_test,
            hue_order=class_names,
            palette="Set2",
            s=72,
            edgecolor="black",
            linewidth=0.8,
            legend=False,
            ax=ax,
        )
        ax.set_title(name)
        ax.set_xlabel("x_km")
        ax.set_ylabel("y_km")

    axes[-1].axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.suptitle("Decision boundaries: train points are small, test points have black edges", y=1.02)
    plt.tight_layout()
    plt.show()

plot_decision_boundaries(trained_classifiers, X_train, y_train, X_test, y_test)


### Student task

Choose two classifiers and compare them:

- Which one has the smoother boundary?
- Which one seems more likely to overfit this small dataset?
- For SVM, change `C` from `3.0` to `0.5` and then to `20.0`. What changes?
- For the decision tree, change `max_depth`. What changes?


## 5. Inspect the best classifier

Use macro F1 to select a model. Macro F1 gives every class the same importance, which is useful when classes have different sample sizes.


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_classifiers[best_model_name]
y_best = best_model.predict(X_test)

print(f"Best model by macro F1: {best_model_name}\n")
print(classification_report(y_test, y_best))

fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_best,
    xticks_rotation=35,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title(f"Confusion matrix: {best_model_name}")
plt.tight_layout()
plt.show()


### Optional feature-engineering task

Repeat the classification with all features:

```python
all_features = spatial_features + attribute_features
X = urban_df[all_features]
```

Do the scores improve? Which model benefits most from the additional variables?


## 6. Prepare the same data for clustering

Clustering is unsupervised: the algorithm does not receive the `zone_type` label.

We still keep the known labels aside so that we can evaluate whether the discovered groups resemble the generated classes. In real exploratory work, this ground truth is often unavailable.


In [ ]:
cluster_features = ["x_km", "y_km", "road_density", "green_share", "transit_access"]
X_cluster = urban_df[cluster_features]
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

true_label_ids = LabelEncoder().fit_transform(urban_df["zone_type"])
print("Clustering features:", cluster_features)


## 7. Run multiple clustering methods

The methods represent the clustering families from the lecture:

- K-Means: centroid-based, requires `k`
- Agglomerative clustering: hierarchical, here also cut to `k=4`
- GMM / EM: distribution-based, assigns points to Gaussian components
- DBSCAN: density-based, can mark points as noise with label `-1`


In [ ]:
clusterers = {
    "K-Means (k=4)": KMeans(n_clusters=4, n_init=20, random_state=RANDOM_STATE),
    "Agglomerative (k=4)": AgglomerativeClustering(n_clusters=4, linkage="ward"),
    "GMM / EM (k=4)": GaussianMixture(n_components=4, covariance_type="full", random_state=RANDOM_STATE),
    "DBSCAN": DBSCAN(eps=0.72, min_samples=6),
}

cluster_labels = {}
cluster_scores = []

for name, method in clusterers.items():
    labels = method.fit_predict(X_cluster_scaled)
    cluster_labels[name] = labels

    n_clusters = len(set(labels) - {-1})
    has_valid_silhouette = len(set(labels)) > 1 and n_clusters > 0
    silhouette = silhouette_score(X_cluster_scaled, labels) if has_valid_silhouette else np.nan
    ari = adjusted_rand_score(true_label_ids, labels)

    cluster_scores.append(
        {
            "method": name,
            "clusters_found": n_clusters,
            "noise_points": int(np.sum(labels == -1)),
            "silhouette": silhouette,
            "adjusted_rand_index": ari,
        }
    )

cluster_scores_df = pd.DataFrame(cluster_scores).sort_values("adjusted_rand_index", ascending=False)
display(cluster_scores_df)


## 8. Side-by-side clustering visualization

All methods below use the same input points. Only the clustering algorithm changes.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)
axes = axes.ravel()

for ax, (name, labels) in zip(axes, cluster_labels.items()):
    plot_df = urban_df.copy()
    plot_df["cluster"] = labels.astype(str)
    plot_df["is_noise"] = labels == -1

    non_noise = plot_df[~plot_df["is_noise"]]
    noise = plot_df[plot_df["is_noise"]]

    sns.scatterplot(
        data=non_noise,
        x="x_km",
        y="y_km",
        hue="cluster",
        palette="tab10",
        s=50,
        edgecolor="white",
        linewidth=0.45,
        legend=False,
        ax=ax,
    )
    if not noise.empty:
        ax.scatter(
            noise["x_km"],
            noise["y_km"],
            c="black",
            marker="x",
            s=55,
            label="noise",
        )

    score_row = cluster_scores_df[cluster_scores_df["method"] == name].iloc[0]
    ax.set_title(
        f"{name}\nclusters={score_row['clusters_found']}, "
        f"noise={score_row['noise_points']}, ARI={score_row['adjusted_rand_index']:.2f}"
    )
    ax.set_xlabel("Local x coordinate (km)")
    ax.set_ylabel("Local y coordinate (km)")

fig.suptitle("Clustering methods on the same small spatial dataset", y=1.02)
plt.tight_layout()
plt.show()


### Student task

Compare the four clustering plots:

- Which method best captures the corridor-like class?
- Which methods force every point into a cluster?
- Which method can mark noise points?
- Change `eps` in DBSCAN to `0.55`, `0.85`, and `1.10`. What happens to the number of clusters and noise points?
- Change `k` / `n_components` from `4` to `3` or `5`. Which visual result is more plausible?


## 9. Link classification and clustering back to the lecture

Use this checklist when writing a short conclusion:

- Supervised learning uses labeled examples and can be evaluated on a held-out test set.
- SVM focuses on a separating boundary and margin; the kernel controls boundary flexibility.
- Decision trees create explicit rules; ensembles such as random forest and gradient boosting usually reduce single-tree weaknesses.
- Unsupervised clustering searches for structure without seeing the class labels.
- K-Means and GMM need a chosen number of groups; DBSCAN uses density parameters instead.
- The best clustering method depends strongly on cluster shape, density, scaling, and the purpose of the analysis.
